In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import yaml
import torch

BASE_PATH = "/content/drive/MyDrive/mosquito-robustness"

with open(os.path.join(BASE_PATH, "configs/config.yaml"), "r") as f:
    config = yaml.safe_load(f)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", DEVICE)

Using device: cpu


In [4]:
import torch.nn as nn
from torchvision import models

In [5]:
!unzip -q "/content/drive/MyDrive/images.zip" -d "/content/"
train_dir = "/content/images/training"
num_classes = len(os.listdir(train_dir))

print("Number of classes:", num_classes)

Number of classes: 6


In [5]:
#efficientnet
def get_efficientnet(num_classes):
    model = models.efficientnet_b0(weights="IMAGENET1K_V1")

    in_features = model.classifier[1].in_features

    # Replace classifier
    model.classifier[1] = nn.Linear(in_features, num_classes)

    # Freeze backbone
    for param in model.features.parameters():
        param.requires_grad = False

    return model, in_features

In [6]:
def get_resnet(num_classes):
    model = models.resnet50(weights="IMAGENET1K_V1")

    in_features = model.fc.in_features

    model.fc = nn.Linear(in_features, num_classes)

    # Freeze backbone
    for name, param in model.named_parameters():
        if "fc" not in name:
            param.requires_grad = False

    return model, in_features

In [7]:
def get_swin(num_classes):
    model = models.swin_v2_t(weights="IMAGENET1K_V1")

    in_features = model.head.in_features

    model.head = nn.Linear(in_features, num_classes)

    # Freeze backbone
    for name, param in model.named_parameters():
        if "head" not in name:
            param.requires_grad = False

    return model, in_features

In [15]:
class ModelWithEmbedding(nn.Module):
    def __init__(self, model, model_name):
        super().__init__()
        self.model = model
        self.model_name = model_name

    def forward(self, x):
        if self.model_name == "resnet":
            # Extract features before FC
            features = self.model.avgpool(self.model.layer4(
                        self.model.layer3(
                        self.model.layer2(
                        self.model.layer1(
                        self.model.maxpool(
                        self.model.relu(
                        self.model.bn1(
                        self.model.conv1(x)))))))))
            features = torch.flatten(features, 1)
            logits = self.model.fc(features)

        elif self.model_name == "efficientnet":
            features = self.model.features(x)
            features = self.model.avgpool(features)
            features = torch.flatten(features, 1)
            logits = self.model.classifier(features)

        elif self.model_name == "swin":

            x=self.model.features(x)

            x=self.model.norm(x)

            features = x.mean(dim=(1,2))

            logits = self.model.head(features)
        return features, logits

In [16]:
eff_model, eff_dim = get_efficientnet(num_classes)
res_model, res_dim = get_resnet(num_classes)
swin_model, swin_dim = get_swin(num_classes)

eff_model = ModelWithEmbedding(eff_model, "efficientnet").to(DEVICE)
res_model = ModelWithEmbedding(res_model, "resnet").to(DEVICE)
swin_model = ModelWithEmbedding(swin_model, "swin").to(DEVICE)

print("Models initialized.")

Models initialized.


In [17]:
#sanity check
dummy = torch.randn(1, 3, 192, 192).to(DEVICE)

for name, model in zip(
    ["EfficientNet", "ResNet", "Swin"],
    [eff_model, res_model, swin_model]
):
    features, logits = model(dummy)
    print(f"{name} → features: {features.shape}, logits: {logits.shape}")

EfficientNet → features: torch.Size([1, 1280]), logits: torch.Size([1, 6])
ResNet → features: torch.Size([1, 2048]), logits: torch.Size([1, 6])
>>> ENTERED SWIN BLOCK <<<
After features: torch.Size([1, 6, 6, 768])
After norm: torch.Size([1, 6, 6, 768])
After mean: torch.Size([1, 768])
Swin → features: torch.Size([1, 768]), logits: torch.Size([1, 6])
